# AUDUSD V. NZDUSD

### IMPORT

In [ ]:
import os, sys, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from arch import arch_model

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", module="statsmodels.tsa.base.tsa_model") # Added to silence the frequency warning
sys.path.append(os.path.abspath('../scripts'))

from spread import SPREAD
from screener import SCREENER
from engine import ENGINE
from backtester import BACKTESTER
from tearsheet import TEARSHEET
from descriptive import DESCRIPTIVE

### PARAMS

In [ ]:
# --- Block 1 & 2: Base Windows (Kept as sensible defaults) ---
TRAIN_DAYS = 30
COINT_WINDOW = 300          # Captures the short-term microstructure beta
Z_WINDOW = 100              # Fast rolling mean to fade bid/ask bounces

# --- Block 3: HMM Architecture (Locked from Synthetic Sandbox) ---
K_REGIMES = 2               # Pure binary states
WINSORIZE_STD = 4.0         # UPDATED: Caps real-world kurtosis to prevent HMM panic
SCALING = 10000             # UPDATED: Wide scaling for statsmodels solver stability

# --- Block 4: Execution Logic (Failed Validation Params) ---
ENTRY_Z = 1.600             
EXIT_Z = -0.200             
DANGER_THRESHOLD = 0.600    
AR_LIMIT = 0.990

In [ ]:
eda_months = ["202401", "202402"]
eda_files = [
    [f"../data/processed/audusd_dukascopy_ask_{m}.parquet" for m in eda_months],
    [f"../data/processed/audusd_dukascopy_bid_{m}.parquet" for m in eda_months],
    [f"../data/processed/nzdusd_dukascopy_ask_{m}.parquet" for m in eda_months],
    [f"../data/processed/nzdusd_dukascopy_bid_{m}.parquet" for m in eda_months],
]

# 2. Build 24-hours for the EDA
builder_eda = SPREAD(agg_type='volume', threshold=1000, active_hours=(0, 24)) 
df_raw_eda = builder_eda.build(eda_files)

# 3. Run Descriptive Stats to see the Bar Chart
eda = DESCRIPTIVE(df_raw_eda, "AUD/USD", "NZD/USD")
eda.generate_full_eda()

### DATA

In [ ]:
months = [
    "202401", "202402", "202403", "202404", "202405", "202406",
    "202407", "202408", "202409", "202410", "202411", "202412",
    "202501", "202502", "202503", "202504", "202505", "202506",
    "202507", "202508", "202509", "202510", "202511", "202512"
]

my_files = [
    [f"../data/processed/audusd_dukascopy_ask_{m}.parquet" for m in months],
    [f"../data/processed/audusd_dukascopy_bid_{m}.parquet" for m in months],
    [f"../data/processed/nzdusd_dukascopy_ask_{m}.parquet" for m in months],
    [f"../data/processed/nzdusd_dukascopy_bid_{m}.parquet" for m in months],
]

### SPREAD

In [ ]:
builder = SPREAD(agg_type='volume', threshold=1000, active_hours=(8, 16)) # Build all hours first
df = builder.build(my_files)
builder.plot_diagnostics()

### DESCRIPTIVE

In [ ]:
eda = DESCRIPTIVE(df, name_a="AUDUSD", name_b="NZDUSD")
eda.generate_full_eda()

### SCREENER

In [ ]:
screener = SCREENER(df['Asset_A'], df['Asset_B'])
p_val, hl = screener.generate_report(rolling_window=2000, rolling_step=200)

### MORE IMPORTS

In [ ]:
import importlib
import engine, backtester, tearsheet
importlib.reload(engine)
importlib.reload(backtester)
importlib.reload(tearsheet)

from engine import ENGINE
from backtester import BACKTESTER
from tearsheet import TEARSHEET

### LIVE TRADING

In [ ]:
live_trading_data, df_params = ENGINE.walk_forward(
    df=df, 
    train_days=TRAIN_DAYS, 
    coint_window=COINT_WINDOW, 
    z_window=Z_WINDOW,
    k_regimes=K_REGIMES,           
    winsorize_std=WINSORIZE_STD,
    scaling=SCALING             # <--- CRITICAL FIX: Added this line
)

### BACKTESTER

In [ ]:
bt = BACKTESTER(live_trading_data)
results_df = bt.run(
    base_z=ENTRY_Z,
    exit_z=EXIT_Z,
    danger_threshold=DANGER_THRESHOLD,   
    ar_limit=AR_LIMIT,
    fee_bps=0.5,
    slippage_mode='half_spread'
)

### TEARSHEET

In [ ]:
ts = TEARSHEET(results_df, df_params=df_params)
ts.generate_report()
ts.plot_performance()
ts.plot_positions_and_regimes()
ts.plot_markov_dynamics()